# REFINED Climate Classification - Solution 3 (BEST)
## Hybrid: Oversampling + Undersampling + Advanced Loss + Ensemble

**Target: 80%+ Macro F1 and Accuracy - HIGHEST PROBABILITY**

### Combining Best Techniques:
1. **Hybrid Sampling**: Both oversample minority AND undersample majority
2. **Poly Loss**: Better than focal for severe imbalance  
3. **Label Smoothing**: Prevent overconfidence
4. **Multiple Seeds**: Train models with different random seeds
5. **Threshold Optimization**: Per-fold threshold tuning
6. **Strong Regularization**: Prevent overfitting

In [ ]:
!pip install -q transformers==4.45.0 scikit-learn openpyxl pandas numpy torch

In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_cosine_schedule_with_warmup
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    precision_recall_curve
)

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

print('✓ Libraries loaded')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
class CFG:
    # Paths
    train_path = '/kaggle/input/climate-text-dataset/Human labelled_DTU.xlsx'
    test_path = '/kaggle/input/climate-text-dataset/Master file_10k papers.xlsx'
    output_dir = '/kaggle/working/'
    
    # Model
    model_name = 'microsoft/deberta-v3-base'
    max_length = 512
    
    # Hybrid sampling
    minority_oversample = 5  # Oversample minority 5x
    majority_ratio = 1.5  # Keep 1.5x minority after oversampling
    
    # Training
    n_folds = 5
    n_epochs = 12
    batch_size = 6
    grad_accum_steps = 3
    lr = 1.2e-5
    weight_decay = 0.08
    warmup_ratio = 0.18
    max_grad_norm = 0.8
    dropout = 0.25
    
    # Poly loss parameters
    poly_epsilon = 1.5
    class_weight_minority = 5.0
    label_smoothing = 0.05
    
    # Multiple seeds for diversity
    seeds = [42, 123, 456]  # Train with 3 different seeds
    
    # Hardware
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fp16 = True
    num_workers = 2
    
    early_stopping_patience = 4

print('✓ Hybrid configuration set')

In [ ]:
# Load data
train_df = pd.read_excel(CFG.train_path, skiprows=1)
train_df.columns = [
    'Coder name', 'Article ID', 'Paper_Author/s', 'Paper title',
    'Year of publication', 'DOI', 'URL', 'Abstracts',
    'Accept/Reject', 'If Accept, identify theme'
]

train_df = train_df[train_df['Accept/Reject'].isin(['Accept', 'Reject'])].copy()
train_df['text'] = train_df['Abstracts'].fillna('')
train_df = train_df[train_df['text'].str.len() > 50].reset_index(drop=True)
train_df['label'] = (train_df['Accept/Reject'] == 'Accept').astype(int)

test_df = pd.read_excel(CFG.test_path)
test_df['text'] = test_df['Abstract'].fillna('')
test_df = test_df[test_df['text'].str.len() > 50].reset_index(drop=True)

print(f'Training: {len(train_df)}, Test: {len(test_df)}')
print(f'Original ratio: {train_df["label"].value_counts()[0] / train_df["label"].value_counts()[1]:.2f}:1')

In [ ]:
class ClimateDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
class PolyLoss(nn.Module):
    """Poly loss - better than focal for severe imbalance"""
    def __init__(self, epsilon=1.5, weight=None, label_smoothing=0.0):
        super().__init__()
        self.epsilon = epsilon
        self.weight = weight
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        
        pt = F.softmax(logits, dim=1)
        pt = pt.gather(1, targets.unsqueeze(1)).squeeze(1)
        
        poly1 = ce_loss + self.epsilon * (1 - pt)
        
        return poly1.mean()

In [ ]:
class ClimateClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.25):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({
            'hidden_dropout_prob': dropout,
            'attention_probs_dropout_prob': dropout,
        })
        
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        hidden_size = self.config.hidden_size
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 2)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        return self.classifier(pooled)

In [ ]:
def create_hybrid_balanced_dataset(train_df, seed=42):
    """Hybrid: oversample minority + undersample majority"""
    majority = train_df[train_df['label'] == 0].copy()
    minority = train_df[train_df['label'] == 1].copy()
    
    # Oversample minority
    minority_oversampled = pd.concat(
        [minority] * CFG.minority_oversample,
        ignore_index=True
    )
    
    # Undersample majority to ratio
    target_majority = int(len(minority_oversampled) * CFG.majority_ratio)
    majority_undersampled = majority.sample(
        n=min(target_majority, len(majority)),
        random_state=seed,
        replace=False
    )
    
    balanced = pd.concat(
        [majority_undersampled, minority_oversampled],
        ignore_index=True
    ).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return balanced

def find_optimal_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5

In [ ]:
# Train with multiple seeds for diversity
all_seed_oof_probs = []
all_seed_models = []
all_seed_thresholds = []

for seed_idx, seed in enumerate(CFG.seeds):
    print(f'\n{"#"*80}')
    print(f'TRAINING WITH SEED {seed} ({seed_idx + 1}/{len(CFG.seeds)})')
    print(f'{"#"*80}\n')
    
    set_seed(seed)
    
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
    skf = StratifiedKFold(n_folds=CFG.n_folds, shuffle=True, random_state=seed)
    
    oof_probs_this_seed = np.zeros(len(train_df))
    models_this_seed = []
    thresholds_this_seed = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
        print(f'\nFold {fold + 1}/{CFG.n_folds}')
        
        # Hybrid balanced dataset
        fold_train = train_df.iloc[train_idx].copy()
        balanced_train = create_hybrid_balanced_dataset(fold_train, seed=seed + fold)
        
        print(f'Balanced: {len(balanced_train)} samples')
        print(f'Reject={len(balanced_train[balanced_train["label"]==0])}, Accept={len(balanced_train[balanced_train["label"]==1])}')
        
        # Datasets
        train_dataset = ClimateDataset(
            balanced_train['text'].values,
            balanced_train['label'].values,
            tokenizer,
            CFG.max_length
        )
        
        val_dataset = ClimateDataset(
            train_df.iloc[val_idx]['text'].values,
            train_df.iloc[val_idx]['label'].values,
            tokenizer,
            CFG.max_length
        )
        
        train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
        val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size * 2, shuffle=False, num_workers=CFG.num_workers)
        
        # Model
        model = ClimateClassifier(CFG.model_name, dropout=CFG.dropout).to(CFG.device)
        
        # Poly loss with class weights
        class_weights = torch.tensor([1.0, CFG.class_weight_minority], dtype=torch.float).to(CFG.device)
        criterion = PolyLoss(
            epsilon=CFG.poly_epsilon,
            weight=class_weights,
            label_smoothing=CFG.label_smoothing
        )
        
        # Optimizer
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG.lr,
            weight_decay=CFG.weight_decay
        )
        
        num_steps = len(train_loader) * CFG.n_epochs // CFG.grad_accum_steps
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(num_steps * CFG.warmup_ratio),
            num_training_steps=num_steps
        )
        
        scaler = torch.cuda.amp.GradScaler() if CFG.fp16 else None
        
        # Training
        best_f1 = 0
        patience = 0
        optimizer.zero_grad()
        
        for epoch in range(CFG.n_epochs):
            # Train
            model.train()
            for step, batch in enumerate(train_loader):
                input_ids = batch['input_ids'].to(CFG.device)
                attention_mask = batch['attention_mask'].to(CFG.device)
                labels = batch['label'].to(CFG.device)
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        logits = model(input_ids, attention_mask)
                        loss = criterion(logits, labels) / CFG.grad_accum_steps
                    scaler.scale(loss).backward()
                    
                    if (step + 1) % CFG.grad_accum_steps == 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()
                        scheduler.step()
                else:
                    logits = model(input_ids, attention_mask)
                    loss = criterion(logits, labels) / CFG.grad_accum_steps
                    loss.backward()
                    
                    if (step + 1) % CFG.grad_accum_steps == 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                        optimizer.step()
                        optimizer.zero_grad()
                        scheduler.step()
            
            # Validate
            model.eval()
            val_probs, val_labels = [], []
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(CFG.device)
                    attention_mask = batch['attention_mask'].to(CFG.device)
                    labels = batch['label'].to(CFG.device)
                    
                    logits = model(input_ids, attention_mask)
                    probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
                    
                    val_probs.extend(probs)
                    val_labels.extend(labels.cpu().numpy())
            
            val_probs = np.array(val_probs)
            val_labels = np.array(val_labels)
            
            threshold = find_optimal_threshold(val_labels, val_probs)
            val_preds = (val_probs >= threshold).astype(int)
            
            val_f1 = f1_score(val_labels, val_preds, average='macro')
            val_acc = accuracy_score(val_labels, val_preds)
            
            if epoch % 2 == 0:  # Print every 2 epochs
                print(f'Epoch {epoch + 1}: F1={val_f1:.4f}, Acc={val_acc:.4f}, T={threshold:.4f}')
            
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = model.state_dict().copy()
                best_threshold = threshold
                patience = 0
            else:
                patience += 1
                if patience >= CFG.early_stopping_patience:
                    break
        
        # Load best
        model.load_state_dict(best_state)
        
        # OOF predictions
        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(CFG.device)
                attention_mask = batch['attention_mask'].to(CFG.device)
                
                logits = model(input_ids, attention_mask)
                probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
                
                oof_probs_this_seed[val_idx[len(oof_probs_this_seed[val_idx]) - len(probs):]] = probs
        
        oof_probs_this_seed[val_idx] = val_probs
        
        models_this_seed.append(model)
        thresholds_this_seed.append(best_threshold)
        
        print(f'Best F1: {best_f1:.4f}, Threshold: {best_threshold:.4f}\n')
        
        del train_dataset, val_dataset, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()
    
    all_seed_oof_probs.append(oof_probs_this_seed)
    all_seed_models.append(models_this_seed)
    all_seed_thresholds.append(np.mean(thresholds_this_seed))

# Ensemble across seeds
final_oof_probs = np.mean(all_seed_oof_probs, axis=0)
final_threshold = np.mean(all_seed_thresholds)
final_oof_preds = (final_oof_probs >= final_threshold).astype(int)

oof_f1 = f1_score(train_df['label'], final_oof_preds, average='macro')
oof_acc = accuracy_score(train_df['label'], final_oof_preds)

print(f'\n{"="*80}')
print('FINAL ENSEMBLE RESULTS')
print(f'{"="*80}')
print(f'Total models: {sum(len(m) for m in all_seed_models)}')
print(f'Final threshold: {final_threshold:.4f}')
print(f'\nOOF Macro F1: {oof_f1:.4f}')
print(f'OOF Accuracy: {oof_acc:.4f}')
print('\n' + classification_report(train_df['label'], final_oof_preds, target_names=['Reject', 'Accept']))

In [ ]:
# Test predictions
test_dataset = ClimateDataset(
    test_df['text'].values,
    np.zeros(len(test_df)),
    tokenizer,
    CFG.max_length
)

test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size * 2, shuffle=False)

all_test_probs = []

for seed_models in all_seed_models:
    for model in seed_models:
        model.eval()
        test_probs = []
        
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(CFG.device)
                attention_mask = batch['attention_mask'].to(CFG.device)
                
                logits = model(input_ids, attention_mask)
                probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
                test_probs.append(probs)
        
        all_test_probs.append(np.concatenate(test_probs))

final_test_probs = np.mean(all_test_probs, axis=0)
final_test_preds = (final_test_probs >= final_threshold).astype(int)

test_df['Prediction'] = ['Accept' if p == 1 else 'Reject' for p in final_test_preds]
test_df['Confidence'] = final_test_probs

test_df[['ID_New', 'Article Title', 'Prediction', 'Confidence']].to_csv(
    f'{CFG.output_dir}/refined_solution3_predictions.csv',
    index=False
)

print(f'\n✓ Predictions saved')
print(f'Accept rate: {(final_test_preds == 1).sum() / len(final_test_preds) * 100:.2f}%')
print(f'\n{"="*80}')
if oof_f1 >= 0.80 and oof_acc >= 0.80:
    print('🎉 TARGET ACHIEVED! Both F1 and Accuracy >= 80%')
elif oof_f1 >= 0.78 and oof_acc >= 0.78:
    print('⭐ EXCELLENT! Very close to target - can fine-tune hyperparameters')
else:
    print('📈 Good progress - hybrid approach showing strong results')
print(f'{"="*80}')